# 02 — Data Cleaning

Applies a documented, auditable fix for each of the 6 issues found in `01_eda.ipynb`. The rule for every fix: **never silently delete rows that represent real revenue** — flag them so downstream KPI queries can decide whether to include them, and log the count so the audit trail survives. This is the same logic implemented in `scripts/clean_and_load.py`; this notebook walks through it narratively.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('data/raw')
orders = pd.read_csv(RAW / 'orders.csv')
customers = pd.read_csv(RAW / 'customers.csv')
returns = pd.read_csv(RAW / 'returns.csv')
regions = pd.read_csv(RAW / 'regions.csv')
print(f'Starting rows: {len(orders):,}')

Starting rows: 59,584


### Fix 1 — drop exact duplicate order rows (keep first occurrence)

In [2]:
compare_cols = [c for c in orders.columns if c != 'order_id']
before = len(orders)
orders = orders.drop_duplicates(subset=compare_cols, keep='first')
print(f'Removed {before - len(orders)} duplicate rows. Remaining: {len(orders):,}')

Removed 459 duplicate rows. Remaining: 59,125


### Fix 2 — flag guest checkouts instead of dropping them

They're real revenue with no customer to attribute it to. Kept in every revenue KPI, excluded from every *customer-level* metric (RFM, CLV, churn model).

In [3]:
orders['is_guest_checkout'] = orders['customer_id'].isna()
print(f"{orders['is_guest_checkout'].sum()} orders flagged as guest checkouts.")

1846 orders flagged as guest checkouts.


### Fix 3 — standardize channel labels to 6 canonical values

In [4]:
CANONICAL = {
    'PAID SEARCH': 'Paid Search', 'paid search': 'Paid Search', 'PAID_SEARCH': 'Paid Search', 'Paid_Search': 'Paid Search',
    'SOCIAL': 'Social', 'social': 'Social',
    'ORGANIC': 'Organic', 'organic': 'Organic',
    'EMAIL': 'Email', 'email': 'Email',
    'AFFILIATE': 'Affiliate', 'affiliate': 'Affiliate',
    'DIRECT': 'Direct', 'direct': 'Direct',
}
before_n = orders['channel'].nunique()
orders['channel'] = orders['channel'].replace(CANONICAL)
print(f"Channel labels: {before_n} raw variants -> {orders['channel'].nunique()} canonical values")
orders['channel'].value_counts()

Channel labels: 19 raw variants -> 6 canonical values


channel
Paid Search    17898
Social         14075
Organic        10695
Email           5882
Affiliate       5843
Direct          4732
Name: count, dtype: int64

### Fix 4 — flag (don't delete) rows with invalid price

These 187 rows have no reconstructable transaction value — there's no reliable way to impute a price a pricing-sync glitch corrupted. They're flagged `is_valid_price = False` and every revenue/margin query in this project filters on that flag.

In [5]:
orders['is_valid_price'] = orders['unit_price'] > 0
print(f"{(~orders['is_valid_price']).sum()} rows flagged invalid (excluded from revenue KPIs).")

187 rows flagged invalid (excluded from revenue KPIs).


### Fix 5 — map missing `region_id` to an explicit 'Unknown' region

Rather than imputing a guessed region (which would quietly bias the regional-performance analysis), these are mapped to region_id = -1 / 'Unknown / Unmapped' so they're visible as their own category and can be excluded from regional rankings without being lost from company-wide totals.

In [6]:
orders['region_id'] = orders['region_id'].fillna(-1).astype(int)
print(f"{(orders['region_id'] == -1).sum()} orders mapped to Unknown region.")

1430 orders mapped to Unknown region.


### Fix 6 — cap refunds at the original order value

A refund can never legitimately exceed what the customer paid. This is a business-rule violation, not ambiguous data — capped at `revenue`, and the count of affected rows is logged so finance can be told the refund system has a bug independent of this analysis.

In [7]:
returns_chk = returns.merge(orders[['order_id','revenue']], on='order_id', how='left')
# drop returns whose order was removed as a duplicate above
returns_chk = returns_chk[returns_chk['order_id'].isin(orders['order_id'])]
over_mask = returns_chk['refund_amount'] > returns_chk['revenue']
print(f'{over_mask.sum()} refunds capped at original order revenue.')
returns_chk.loc[over_mask, 'refund_amount'] = returns_chk.loc[over_mask, 'revenue']
returns_clean = returns_chk.drop(columns=['revenue'])

32 refunds capped at original order revenue.


### Save cleaned tables (single `data/processed/` directory — no duplicate folders)

In [8]:
Path('data/processed').mkdir(exist_ok=True)
orders.to_csv('data/processed/orders_clean.csv', index=False)
returns_clean.to_csv('data/processed/returns_clean.csv', index=False)
print('Saved cleaned orders and returns.')
print(f"Final orders table: {len(orders):,} rows, {orders['is_valid_price'].sum():,} with usable price")

Saved cleaned orders and returns.
Final orders table: 59,125 rows, 58,938 with usable price
